# Purringer i Altinn

> Hensikten med scriptet er å bruke delreg 24xx til å hente ut informasjon om hvilke private foretak som har levert svar på skjema 39, og om mulig legge inn en variabel i 20877xx for purringsvarsel via Altinn


#### Gjenstår:

- [ ] Finne hvilke enheter som har svart. Sjekk ut `KVITT_TYPE`
- [ ] Gjøre endringer i delreg 20877xx på en slik måte at Altinn sender purring til korrekt foretak

In [ ]:
import numpy as np
import pandas as pd
import cx_Oracle
from db1p import query_db1p
import getpass
import os
import requests
from pandas.io.json import json_normalize #package for flattening json in pandas df

# Fjerner begrensning på antall rader og kolonner som vises av gangen
pd.set_option("display.max_columns", None)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)

# Unngå standardform i output
pd.set_option('display.float_format', lambda x: '%.0f' % x)

In [ ]:
conn = cx_Oracle.connect(getpass.getuser()+"/"+getpass.getpass(prompt='Oracle-passord: ')+"@DB1P")

## Delreg 24xx

In [ ]:
sporring = f"""
    SELECT *
    FROM DSBBASE.DLR_ENHET_I_DELREG
    WHERE DELREG_NR IN ('2422')
"""
SFU_data = pd.read_sql_query(sporring, conn)
print(f"Rader:    {SFU_data.shape[0]}\nKolonner: {SFU_data.shape[1]}")
SFU_data.info()


In [ ]:
def print_arr(arr):
    for x in arr:
        print(x)

In [ ]:
kvitt_arr=[]
for x in SFU_data.columns:
    if x.__contains__("KVITT"):
        kvitt_arr.append(x)

In [ ]:
dato_arr=[]
for x in SFU_data.columns:
    if x.__contains__("DATO"):
        dato_arr.append(x)

In [ ]:
hj_arr=[]
for x in SFU_data.columns:
    if x.__contains__("H_VAR") or x.__contains__("SN"):
        hj_arr.append(x)

In [ ]:
navn_arr = ['ORGNR', 'ORGNR_FORETAK', 'NAVN']

In [ ]:
SFU_priv = SFU_data[SFU_data['H_VAR2_A']=='PRIVAT']

In [ ]:
SFU_priv.shape

In [ ]:
SFU_priv[navn_arr + dato_arr].info()

In [ ]:
SFU_priv[navn_arr + kvitt_arr].info()

In [ ]:
navn_arr

In [ ]:
SFU_priv[['KVITT_TYPE'] + navn_arr]

In [ ]:
SFU_priv[navn_arr + hj_arr]

In [ ]:
# print_arr(SFU_data.columns)

## Delreg 20877xx

In [ ]:
def print_arr(arr):
    for x in arr:
        print(x)

In [ ]:
sporring = f"""
    SELECT *
    FROM DSBBASE.DLR_ENHET_I_DELREG
    WHERE DELREG_NR IN ('2087722')
"""
altinn = pd.read_sql_query(sporring, conn)
print(f"Rader:    {altinn.shape[0]}\nKolonner: {altinn.shape[1]}")

In [ ]:
kvitt_arr=[]
for x in altinn.columns:
    if x.__contains__("KVITT"):
        kvitt_arr.append(x)

In [ ]:
dato_arr=[]
for x in altinn.columns:
    if x.__contains__("DATO"):
        dato_arr.append(x)

In [ ]:
hj_arr=[]
for x in altinn.columns:
    if x.__contains__("H_VAR") or x.__contains__("SN"):
        hj_arr.append(x)

In [ ]:
navn_arr = ['ORGNR', 'ORGNR_FORETAK', 'NAVN']

In [ ]:
altinn[navn_arr + kvitt_arr].dropna()

In [ ]:
# altinn[['KVITT_TYPE'] + navn_arr]

In [ ]:
# altinn[navn_arr + hj_arr]